# 05 数据清洗、对齐与收益率

## 5.1 本章要解决什么问题

第 04 章已经把 AKShare 获取到的 ETF 日线行情标准化并缓存到了 `data/sample/`。本章不再重复联网，而是把这份标准行情变成后续因子、组合和回测可以直接使用的研究数据。

读完并运行本章后，你会得到：

- 清洗后的 `date, code, open, high, low, close, volume, amount` 长表；
- 对齐到共同交易日的 ETF 收盘价矩阵和成交量矩阵；
- 简单收益率矩阵；
- 一份数据质量摘要，用来检查日期覆盖、重复值和缺失值。

## 5.2 前置条件

- 已运行第 04 章，或项目中已经存在 `data/sample/prices.parquet`、`data/sample/calendar.parquet`、`data/sample/assets.parquet`；
- 了解 `pandas` 的 `groupby`、`pivot`、`pct_change` 基本用法。

## 5.3 本章路线

1. 先用手写小例子看懂清洗、去重、对齐和收益率；
2. 再讨论缺失值和 point-in-time 边界；
3. 最后用 `lib.data.transforms` 复现一遍实战数据处理，并把小型结果保存到 `data/processed/` 与 `outputs/results/`。


In [1]:
from pathlib import Path
import sys


def find_project_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "lib").exists() and (candidate / "notebooks").exists():
            return candidate
    raise RuntimeError("请从 pyquant-roadmap 项目目录或 notebooks 目录运行本 notebook")


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

pd.set_option("display.max_rows", 8)
pd.set_option("display.max_columns", 12)

from lib.paths import PROCESSED_DIR, RESULTS_DIR, SAMPLE_DIR

pd.Series(
    {
        "project_root": ".",
        "sample_dir": SAMPLE_DIR.relative_to(PROJECT_ROOT).as_posix(),
        "results_dir": RESULTS_DIR.relative_to(PROJECT_ROOT).as_posix(),
    },
    name="value",
)


project_root                  .
sample_dir          data/sample
results_dir     outputs/results
Name: value, dtype: object

## 5.4 本章在主线中的位置

数据主线是：

`第 04 章 获取并标准化行情 -> 第 05 章 清洗、对齐、收益率 -> 第 06 章 因子构建 -> 第 07 章 因子检查 -> 第 08 章 组合权重 -> 第 09 章 回测`

本章只处理历史行情本身，不在这里做选股，也不在这里合成组合收益。原因是：资产收益率矩阵回答的是“每只 ETF 每天涨跌多少”，组合收益要等权重和交易成本确定后才能计算。


In [2]:
from lib.data.sample import load_sample_assets, load_sample_calendar, load_sample_prices

prices = load_sample_prices()
calendar = load_sample_calendar()
assets = load_sample_assets()

display(assets)
display(prices.groupby("code")["date"].agg(["min", "max", "count"]))
display(calendar.head())


,code,name,asset_type,list_date
0,510300,沪深300ETF,ETF,2000-01-01
1,510500,中证500ETF,ETF,2000-01-01
2,159915,创业板ETF,ETF,2000-01-01
3,512100,中证1000ETF,ETF,2000-01-01


,min,max,count
code,,,
159915,2021-01-04,2023-12-29,725
510300,2021-01-04,2023-12-29,725
510500,2021-01-04,2023-12-29,725
512100,2021-01-04,2023-12-29,725


,date,is_open
0,2021-01-04,1
1,2021-01-05,1
2,2021-01-06,1
3,2021-01-07,1
4,2021-01-08,1


## 5.5 手写最小实现：清洗标准行情长表

第 04 章已经做过字段标准化，所以本章不再处理 AKShare 的中文列名。这里的清洗只做四件事：

- `date` 转成真正的日期类型；
- `code` 统一成字符串；
- 价格、成交量、成交额转成数值；
- 同一个 `date, code` 出现重复行时，只保留最后一条。

重复行的处理不是为了“猜哪条一定正确”，而是为了让后续 `pivot` 有唯一键。如果真实项目中重复来自数据源修订，应在数据日志里保留原因。


In [3]:
toy_raw = pd.DataFrame(
    {
        "date": ["2024-01-02", "2024-01-02", "2024-01-02", "2024-01-03", "2024-01-03", "2024-01-04"],
        "code": ["ETF_A", "ETF_A", "ETF_B", "ETF_A", "ETF_B", "ETF_A"],
        "open": ["10.00", "10.10", "20.00", "10.30", "20.40", "10.50"],
        "high": ["10.20", "10.25", "20.30", "10.50", "20.50", "10.70"],
        "low": ["9.90", "10.00", "19.90", "10.10", "20.10", "10.30"],
        "close": ["10.10", "10.20", "20.20", "10.40", "20.30", "10.60"],
        "volume": ["1000", "1200", "2000", "1500", "1800", "1600"],
        "amount": [10100, 12240, 40400, 15600, 36540, 16960],
    }
)


def clean_prices_manual(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["_row_order"] = range(len(out))
    out["date"] = pd.to_datetime(out["date"], errors="coerce")
    out["code"] = out["code"].astype(str)
    for col in ["open", "high", "low", "close", "volume", "amount"]:
        out[col] = pd.to_numeric(out[col], errors="coerce")
    out = out.dropna(subset=["date", "code"])
    out = out.sort_values(["date", "code", "_row_order"])
    out = out.drop_duplicates(["date", "code"], keep="last")
    return out.drop(columns="_row_order").sort_values(["date", "code"]).reset_index(drop=True)


toy_clean = clean_prices_manual(toy_raw)
toy_clean


,date,code,open,high,low,close,volume,amount
0,2024-01-02,ETF_A,10.1,10.25,10.0,10.2,1200,12240
1,2024-01-02,ETF_B,20.0,20.30,19.9,20.2,2000,40400
2,2024-01-03,ETF_A,10.3,10.50,10.1,10.4,1500,15600
3,2024-01-03,ETF_B,20.4,20.50,20.1,20.3,1800,36540
4,2024-01-04,ETF_A,10.5,10.70,10.3,10.6,1600,16960


## 5.6 手写最小实现：对齐到共同交易日

横截面因子、组合权重和回测通常需要同一天看到同一组资产。对 ETF 池来说，最保守的起步方式是：

1. 先用交易日历过滤真实开市日；
2. 再只保留所有目标 ETF 都有行情的日期；
3. 不在这一步把缺失价格伪装成正常交易价格。

这样做会少用一些样本，但能让第 06 章的矩阵计算更清楚。


In [4]:
toy_calendar = pd.DataFrame(
    {
        "date": pd.to_datetime(["2024-01-02", "2024-01-03", "2024-01-04", "2024-01-05"]),
        "is_open": [1, 1, 1, 0],
    }
)


def align_common_dates_manual(
    prices_df: pd.DataFrame,
    calendar_df: pd.DataFrame,
    codes: list[str],
) -> pd.DataFrame:
    open_dates = pd.to_datetime(calendar_df.loc[calendar_df["is_open"].eq(1), "date"])
    work = prices_df[prices_df["date"].isin(open_dates) & prices_df["code"].isin(codes)].copy()
    counts = work.groupby("date")["code"].nunique()
    common_dates = counts[counts.eq(len(codes))].index
    return work[work["date"].isin(common_dates)].sort_values(["date", "code"]).reset_index(drop=True)


toy_aligned = align_common_dates_manual(toy_clean, toy_calendar, ["ETF_A", "ETF_B"])
display(toy_clean)
display(toy_aligned)


,date,code,open,high,low,close,volume,amount
0,2024-01-02,ETF_A,10.1,10.25,10.0,10.2,1200,12240
1,2024-01-02,ETF_B,20.0,20.30,19.9,20.2,2000,40400
2,2024-01-03,ETF_A,10.3,10.50,10.1,10.4,1500,15600
3,2024-01-03,ETF_B,20.4,20.50,20.1,20.3,1800,36540
4,2024-01-04,ETF_A,10.5,10.70,10.3,10.6,1600,16960


,date,code,open,high,low,close,volume,amount
0,2024-01-02,ETF_A,10.1,10.25,10.0,10.2,1200,12240
1,2024-01-02,ETF_B,20.0,20.30,19.9,20.2,2000,40400
2,2024-01-03,ETF_A,10.3,10.50,10.1,10.4,1500,15600
3,2024-01-03,ETF_B,20.4,20.50,20.1,20.3,1800,36540


## 5.7 手写最小实现：收盘价、成交量和简单收益率矩阵

长表适合存储和追加，矩阵适合做计算。第 06 章的动量、波动率、均线偏离都会基于矩阵计算。

简单收益率的公式是：

`ret[t] = close[t] / close[t-1] - 1`

第一行没有前一日价格，本章为了后续资产收益矩阵容易落盘，把第一行填成 `0.0`。但如果中间出现缺失，本章不会默认填成 0，因为那会掩盖数据问题。


In [5]:
def make_matrix_manual(prices_df: pd.DataFrame, value_col: str) -> pd.DataFrame:
    return prices_df.pivot(index="date", columns="code", values=value_col).sort_index().astype(float)


def simple_returns_manual(close_matrix: pd.DataFrame) -> pd.DataFrame:
    out = close_matrix.pct_change(fill_method=None)
    if not out.empty:
        out.iloc[0] = out.iloc[0].fillna(0.0)
    return out


toy_close = make_matrix_manual(toy_aligned, "close")
toy_volume = make_matrix_manual(toy_aligned, "volume")
toy_returns = simple_returns_manual(toy_close)

display(toy_close)
display(toy_volume)
display(toy_returns.round(4))


code,ETF_A,ETF_B
date,,
2024-01-02,10.2,20.2
2024-01-03,10.4,20.3


code,ETF_A,ETF_B
date,,
2024-01-02,1200.0,2000.0
2024-01-03,1500.0,1800.0


code,ETF_A,ETF_B
date,,
2024-01-02,0.0000,0.000
2024-01-03,0.0196,0.005


## 5.8 缺失值处理：先解释，再填充

缺失值不能“一把填平”。至少先区分三种情况：

- 停牌或不可交易：可能可以用上一收盘价估值，但成交量应表现为不可交易；
- 数据源漏数：应回源修复或剔除，不应该悄悄填充；
- 滚动窗口不够：早期样本天然为空，保留为空更诚实。

下面用一列人为缺口说明：`pct_change(fill_method=None)` 会把缺失影响留在收益率里；有限前向填充只适合你已经确认缺口代表“估值沿用上一收盘价”的场景。


In [6]:
gap_close = pd.DataFrame(
    {
        "ETF_A": [10.0, 10.5, np.nan, 11.2],
        "ETF_B": [20.0, 20.2, 20.0, 20.6],
    },
    index=pd.to_datetime(["2024-01-02", "2024-01-03", "2024-01-04", "2024-01-05"]),
)

raw_ret = gap_close.pct_change(fill_method=None)
valuation_close = gap_close.ffill(limit=1)
valuation_ret = valuation_close.pct_change(fill_method=None)

missing_example = pd.concat(
    {
        "raw_close": gap_close["ETF_A"],
        "raw_ret": raw_ret["ETF_A"],
        "ffill_close_limit1": valuation_close["ETF_A"],
        "ffill_ret": valuation_ret["ETF_A"],
    },
    axis=1,
)
missing_example.round(4)


,raw_close,raw_ret,ffill_close_limit1,ffill_ret
2024-01-02,10.0,NaN,10.0,NaN
2024-01-03,10.5,0.05,10.5,0.0500
2024-01-04,NaN,NaN,10.5,0.0000
2024-01-05,11.2,NaN,11.2,0.0667


## 5.9 Point-in-time 边界：什么时候能看到，什么时候能交易

清洗和对齐不是只为格式好看，更重要的是避免未来函数。日频研究里可以先记一条保守规则：

- 如果信号要等 T 日收盘价出来后才能计算，就不能假设自己在 T 日收盘前已经知道它；
- 如果策略按下一交易日成交，那么未来收益标签至少从 T+1 开始；
- 对财务数据、成分股、停复牌事件等非价格字段，应按公告日、披露日或真实可得时间对齐，而不是按报告期末直接合并。

下面的 `future_return_from_next_session` 表达的是：用 T 日收盘后形成的信号，解释 T+1 到 T+2 的收益。


In [7]:
pit_close = pd.Series(
    [10.0, 10.3, 10.1, 10.6, 10.8],
    index=pd.to_datetime(["2024-01-02", "2024-01-03", "2024-01-04", "2024-01-05", "2024-01-08"]),
    name="ETF_A",
)

pit_frame = pd.DataFrame(
    {
        "close_t": pit_close,
        "signal_1d_mom_at_t_close": pit_close.pct_change(fill_method=None),
        "future_return_from_next_session": pit_close.shift(-2) / pit_close.shift(-1) - 1,
    }
)
pit_frame.round(4)


,close_t,signal_1d_mom_at_t_close,future_return_from_next_session
2024-01-02,10.0,NaN,-0.0194
2024-01-03,10.3,0.0300,0.0495
2024-01-04,10.1,-0.0194,0.0189
2024-01-05,10.6,0.0495,NaN
2024-01-08,10.8,0.0189,NaN


## 5.10 使用 pandas / lib 复现实战处理

前面手写的逻辑在实战中会反复出现，所以项目把稳定部分沉淀到 `lib.data.transforms`。这里直接从模块导入，让后续 notebook 可以复用同一套清洗、对齐和收益率口径。

`pandas` 负责真正的表格运算：

- `groupby` 用来找每个日期覆盖了几只 ETF；
- `pivot` 用来把长表转成矩阵；
- `pct_change(fill_method=None)` 用来计算简单收益率，并避免默认填补缺失价格。

`lib` 只固定本项目的口径：标准字段、共同交易日、第一行收益率填 0、中间缺口保留。


In [8]:
from lib.data.transforms import build_aligned_market_data, clean_price_data, missing_value_report

codes = assets["code"].astype(str).tolist()
market = build_aligned_market_data(prices, calendar=calendar, codes=codes)

before_coverage = clean_price_data(prices).groupby("code")["date"].agg(raw_start="min", raw_end="max", raw_rows="count")
after_coverage = market.prices.groupby("code")["date"].agg(aligned_start="min", aligned_end="max", aligned_rows="count")
coverage = assets.merge(before_coverage, left_on="code", right_index=True).merge(
    after_coverage, left_on="code", right_index=True
)

display(coverage)
display(market.quality)


,code,name,asset_type,list_date,raw_start,raw_end,raw_rows,aligned_start,aligned_end,aligned_rows
0,510300,沪深300ETF,ETF,2000-01-01,2021-01-04,2023-12-29,725,2021-01-04,2023-12-29,725
1,510500,中证500ETF,ETF,2000-01-01,2021-01-04,2023-12-29,725,2021-01-04,2023-12-29,725
2,159915,创业板ETF,ETF,2000-01-01,2021-01-04,2023-12-29,725,2021-01-04,2023-12-29,725
3,512100,中证1000ETF,ETF,2000-01-01,2021-01-04,2023-12-29,725,2021-01-04,2023-12-29,725


,code,start,end,rows,open_na,high_na,low_na,close_na,volume_na,amount_na
0,159915,2021-01-04,2023-12-29,725,0,0,0,0,0,0
1,510300,2021-01-04,2023-12-29,725,0,0,0,0,0,0
2,510500,2021-01-04,2023-12-29,725,0,0,0,0,0,0
3,512100,2021-01-04,2023-12-29,725,0,0,0,0,0,0


In [9]:
print(f"close matrix shape: {market.close.shape}")
print(f"volume matrix shape: {market.volume.shape}")
print(f"returns matrix shape: {market.returns.shape}")

display(market.close.tail())
display(market.volume.tail())
display(market.returns.tail().round(4))


close matrix shape: (725, 4)
volume matrix shape: (725, 4)
returns matrix shape: (725, 4)


code,510300,510500,159915,512100
date,,,,
2023-12-25,3.135,5.163,1.783,2.232
2023-12-26,3.115,5.116,1.761,2.202
2023-12-27,3.125,5.132,1.763,2.215
2023-12-28,3.209,5.235,1.833,2.262
2023-12-29,3.219,5.279,1.842,2.296


code,510300,510500,159915,512100
date,,,,
2023-12-25,10041465.0,3139644.0,5703304.0,2871125.0
2023-12-26,7771693.0,2543510.0,8185801.0,3178739.0
2023-12-27,8361725.0,2568572.0,6536878.0,3695511.0
2023-12-28,23832992.0,3487194.0,15884491.0,4967557.0
2023-12-29,25188735.0,2713357.0,7493274.0,3659420.0


code,510300,510500,159915,512100
date,,,,
2023-12-25,0.0029,-0.0014,0.0017,-0.0018
2023-12-26,-0.0064,-0.0091,-0.0123,-0.0134
2023-12-27,0.0032,0.0031,0.0011,0.0059
2023-12-28,0.0269,0.0201,0.0397,0.0212
2023-12-29,0.0031,0.0084,0.0049,0.0150


## 5.11 质量检查

这一步不是为了追求表面上“没有空值”，而是确认本章的处理口径符合预期：

- `date, code` 不重复；
- 收盘价和成交量矩阵没有缺口；
- 收益率除了第一行以外没有新增缺口；
- 对齐后的日期范围和资产数量与 ETF 池一致。


In [10]:
returns_missing_after_first = 0
if len(market.returns) > 1:
    returns_missing_after_first = int(market.returns.iloc[1:].isna().sum().sum())

quality_checks = pd.Series(
    {
        "aligned_rows": int(len(market.prices)),
        "aligned_dates": int(market.close.shape[0]),
        "aligned_assets": int(market.close.shape[1]),
        "duplicate_date_code": int(market.prices.duplicated(["date", "code"]).sum()),
        "close_missing_cells": int(market.close.isna().sum().sum()),
        "volume_missing_cells": int(market.volume.isna().sum().sum()),
        "returns_missing_after_first_row": returns_missing_after_first,
        "first_date": market.close.index.min().strftime("%Y-%m-%d"),
        "last_date": market.close.index.max().strftime("%Y-%m-%d"),
    }
)
quality_checks


aligned_rows                             2900
aligned_dates                             725
aligned_assets                              4
duplicate_date_code                         0
                                      ...    
volume_missing_cells                        0
returns_missing_after_first_row             0
first_date                         2021-01-04
last_date                          2023-12-29
Length: 9, dtype: object

## 5.12 保存本章产出

本章产出都是小型派生文件，适合保存，方便第 06 章或后续实验直接检查：

- `data/processed/chapter05_aligned_prices.parquet`：共同交易日上的标准 OHLCV 长表；
- `data/processed/chapter05_close_matrix.csv`：收盘价矩阵；
- `data/processed/chapter05_volume_matrix.csv`：成交量矩阵；
- `data/processed/chapter05_returns_matrix.csv`：资产简单收益率矩阵；
- `outputs/results/chapter05_data_quality_summary.csv`：本章质量摘要。


In [11]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

output_paths = {
    "aligned_prices": PROCESSED_DIR / "chapter05_aligned_prices.parquet",
    "close_matrix": PROCESSED_DIR / "chapter05_close_matrix.csv",
    "volume_matrix": PROCESSED_DIR / "chapter05_volume_matrix.csv",
    "returns_matrix": PROCESSED_DIR / "chapter05_returns_matrix.csv",
    "quality_summary": RESULTS_DIR / "chapter05_data_quality_summary.csv",
}

market.prices.to_parquet(output_paths["aligned_prices"], index=False)
market.close.to_csv(output_paths["close_matrix"], index_label="date", encoding="utf-8-sig")
market.volume.to_csv(output_paths["volume_matrix"], index_label="date", encoding="utf-8-sig")
market.returns.to_csv(output_paths["returns_matrix"], index_label="date", encoding="utf-8-sig")
quality_checks.rename_axis("metric").to_frame("value").to_csv(output_paths["quality_summary"], encoding="utf-8-sig")

pd.Series({name: str(path.relative_to(PROJECT_ROOT)) for name, path in output_paths.items()})


aligned_prices       data\processed\chapter05_aligned_prices.parquet
close_matrix               data\processed\chapter05_close_matrix.csv
volume_matrix             data\processed\chapter05_volume_matrix.csv
returns_matrix           data\processed\chapter05_returns_matrix.csv
quality_summary    outputs\results\chapter05_data_quality_summary...
dtype: object

## 5.13 练习：换一个 ETF 子池重新对齐

练习目标：只选择前三只 ETF，重新生成共同交易日矩阵，并检查日期数量是否变化。

思考：如果某只 ETF 上市更晚，`aligned_dates` 会变多还是变少？为什么这比直接前向填充更稳？


In [12]:
exercise_codes = codes[:3]
exercise_market = build_aligned_market_data(prices, calendar=calendar, codes=exercise_codes)

pd.Series(
    {
        "codes": ",".join(exercise_codes),
        "aligned_dates": exercise_market.close.shape[0],
        "aligned_assets": exercise_market.close.shape[1],
        "missing_close_cells": int(exercise_market.close.isna().sum().sum()),
    }
)


codes                  510300,510500,159915
aligned_dates                           725
aligned_assets                            3
missing_close_cells                       0
dtype: object

## 5.14 小结与交接给第 06 章

本章把标准化行情变成了“可以比较”的研究输入：

- 长表负责存储完整 OHLCV；
- 收盘价矩阵负责计算动量、均线和收益率；
- 成交量矩阵负责后续流动性检查；
- 收益率矩阵只是资产收益，不是组合收益；
- point-in-time 边界决定了信号和未来收益标签之间必须错开。

第 06 章会在这个基础上构造因子：动量来自收盘价相对变化，低波动来自收益率滚动标准差，均线偏离来自不同窗口均线的相对位置。也就是说，第 06 章真正消耗的不是“原始接口返回值”，而是本章整理出的、口径一致的价格序列。
